In [2]:
#r "nuget: FSharp.Stats, 0.4.3"
#r "nuget: BioFSharp, 2.0.0-beta5"
#r "nuget: BioFSharp.IO, 2.0.0-beta5"
#r "nuget: Plotly.NET, 2.0.0-preview.16"
#r "nuget: Deedle, 2.5.0"

#r "nuget: Plotly.NET.Interactive, 2.0.0-preview.16"

open Deedle
open BioFSharp
open FSharpAux
open FSharp.Stats
open Plotly.NET


Installed Packages BioFSharp, 2.0.0-beta5 BioFSharp.IO, 2.0.0-beta5 Deedle, 2.5.0 FSharp.Stats, 0.4.3 Plotly.NET, 2.0.0-preview.16 Plotly.NET.Interactive, 2.0.0-preview.16

# NB08b Label efficiency for SDS

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB08b_Label_efficiency_BN.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB08b/NB08b_Label_efficiency_BN.ipynb)

Stable isotopic peptide labeling is the foundation of QconCAT experiments. While an excellent tool when carried out with correctly, it also exposes 
challenges and pitfalls that have to be checked and possibly accounted for. One of these pitfalls is the efficiency with which we labeled 
our QconCAT protein (Why?). In this notebook we will have a look at some high quality peptides selected in the previous notebook and 
illustrate how the label efficiency can be calculated using simulations.  

## I. Reading the data
As promised, we start this notebook with the output of the previous analysis, this notebook assumes that the data from *NB08a Data Access and Quality Control* is stored in a .txt


In [3]:
type PeptideIon = 
    {|
        ProteinGroup    : string  
        Synonyms        : string
        StringSequence  : string
        PepSequenceID   : int
        Charge          : int
    |}

//This is the filepath you chose in *NB08a Data Access and Quality Control*
// let filePath = @"C:\YourPath\testOut.txt"
let filePath = @"C:\YourPath\testOut.txt"

// What is different about this function from the one known from the last notebook?
let qConcatDataFiltered =
    Frame.ReadCsv(path = filePath, separators = "\t")
    // StringSequence is the peptide sequence
    |> Frame.indexRowsUsing (fun os -> 
            let proteinGroup = os.GetAs<string>("ProteinGroup")
            {|
                ProteinGroup    = os.GetAs<string>("ProteinGroup"); 
                Synonyms        = os.GetAs<string>("Synonym")
                StringSequence  = os.GetAs<string>("StringSequence");
                PepSequenceID   = os.GetAs<int>("PepSequenceID");
                Charge          = os.GetAs<int>("Charge");
            |}
        )
    |> Frame.filterRows (fun k s -> k.ProteinGroup |> String.contains "QProt_newPS")

qConcatDataFiltered.ColumnKeys
|> Array.ofSeq


Error: System.IO.FileNotFoundException: Could not find file '/home/paulinehans/Dokumente/BIO-BTE-06-L-7/Notebooks/C:\YourPath\testOut.txt'.
File name: '/home/paulinehans/Dokumente/BIO-BTE-06-L-7/Notebooks/C:\YourPath\testOut.txt'
   at Interop.ThrowExceptionForIoErrno(ErrorInfo errorInfo, String path, Boolean isDirError)
   at Microsoft.Win32.SafeHandles.SafeFileHandle.Open(String path, OpenFlags flags, Int32 mode, Boolean failForSymlink, Boolean& wasSymlink, Func`4 createOpenException)
   at Microsoft.Win32.SafeHandles.SafeFileHandle.Open(String fullPath, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize, UnixFileMode openPermissions, Int64& fileLength, UnixFileMode& filePermissions, Boolean failForSymlink, Boolean& wasSymlink, Func`4 createOpenException)
   at System.IO.Strategies.OSFileStreamStrategy..ctor(String path, FileMode mode, FileAccess access, FileShare share, FileOptions options, Int64 preallocationSize, Nullable`1 unixCreateMo

First we reuse a proved pattern and define a function to manipulate our frame


In [3]:
let sliceQuantColumns quantColID frame = 
    frame
    |> Frame.filterCols (fun ck os -> ck |> String.contains ("." + quantColID))
    |> Frame.mapColKeys (fun ck -> ck.Split('.') |> Array.item 0)


Besides already familiar slices...


In [4]:
let heavy = sliceQuantColumns "Quant_Heavy" qConcatDataFiltered


Error: input.fsx (1,45)-(1,64) typecheck error The value or constructor 'qConcatDataFiltered' is not defined. Maybe you want one of the following:
   ConfidenceInterval

... we can also use this function for information needed to reconstruct isotopic patterns.

## II. Extraction and visualization of measured isotopic envelopes.


In [5]:
let heavyPatternMz = sliceQuantColumns "heavyPatternMz" qConcatDataFiltered
let heavyPatternI  = sliceQuantColumns "heavyPatternI" qConcatDataFiltered


Error: input.fsx (1,57)-(1,76) typecheck error The value or constructor 'qConcatDataFiltered' is not defined. Maybe you want one of the following:
   ConfidenceInterval
input.fsx (2,56)-(2,75) typecheck error The value or constructor 'qConcatDataFiltered' is not defined. Maybe you want one of the following:
   ConfidenceInterval

Now, there's a challenge: The info to reconstruct an isotopic pattern is
separated into two columns, the x component (heavyPatternMz) and the y component (heavyPatternI).
As always, this challenged can be solved using a function! 
Hint: Note how we define a function 'floatArrayOf' that specifies how the string is parsed. 


In [6]:
let getHeavyPatternsInFile fileName = 
    let floatArrayOf s = 
        if String.isNullOrEmpty s then 
            [||]
        else
            s
            |> String.split (';') 
            |> Array.map float
    let mz, intensities =
        heavyPatternMz
        |> Frame.getCol fileName 
        |> Series.mapValues floatArrayOf,
        heavyPatternI
        |> Frame.getCol fileName 
        |> Series.mapValues floatArrayOf
    let zipped = Series.zipInner mz intensities
    zipped

let extractedPatterns = getHeavyPatternsInFile "20210312BN2_U1"


Error: input.fsx (10,9)-(10,23) typecheck error The value or constructor 'heavyPatternMz' is not defined.
input.fsx (13,9)-(13,22) typecheck error The value or constructor 'heavyPatternI' is not defined.

Additionally, we can write two functions to plot the patterns of a peptide. When it comes
to the build the chart (plotIsotopicPattern), things get a little bit trickier, but this is not necessarily your concern. Please inspect the Chart 
created by 'plotIsotopicPatternOf' and write correct descriptions for the x and the y axis. (Fill: |> Chart.withX_AxisStyle "" and |> Chart.withY_AxisStyle "")


In [7]:
let plotIsotopicPattern color mzsAndintensities =
    let min,max =
        mzsAndintensities |> Seq.minBy fst |> fst,
        mzsAndintensities |> Seq.maxBy fst |> fst
    Seq.map (fun (x,y) -> 
        Chart.Line([x;x],[0.;y], ShowLegend = false)
        |> Chart.withLineStyle (Width = 7)
    ) mzsAndintensities
    |> Chart.combine
    |> Chart.withMarkerStyle(Size=0,Color = Color.fromHex (FSharpAux.Colors.toWebColor color))
    |> Chart.withXAxisStyle ("", MinMax = (min - 1., max + 1.))
    |> Chart.withYAxisStyle ""

type ExtractedIsoPattern = 
    {|
        PeptideSequence : PeptideIon
        Charge          : int
        Pattern         : seq<(float*float)>
    |}

let getIsotopicPattern peptideSequence charge =
    let (k,(mzs,intensities)) = 
        extractedPatterns
        |> Series.observations
        |> Seq.find (fun (k,(mzs,intensities)) -> 
                k.StringSequence = peptideSequence && k.Charge = charge
            )
    {|
        PeptideSequence=k
        Charge  = charge
        Pattern = Seq.zip mzs intensities
    |}


Error: input.fsx (16,27)-(16,37) typecheck error The type 'PeptideIon' is not defined.
input.fsx (16,27)-(16,37) typecheck error The type 'PeptideIon' is not defined.
input.fsx (23,9)-(23,26) typecheck error The value or constructor 'extractedPatterns' is not defined. Maybe you want one of the following:
   ExtractedIsoPattern
input.fsx (26,17)-(26,33) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (26,55)-(26,63) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.

In [8]:
let examplePep1 = getIsotopicPattern "DTDILAAFR" 2


Error: input.fsx (1,19)-(1,37) typecheck error The value or constructor 'getIsotopicPattern' is not defined. Maybe you want one of the following:
   getCvOfReplicates
   getStDevOfReplicates

In [9]:
plotIsotopicPattern FSharpAux.Colors.Table.Office.blue examplePep1.Pattern


Error: input.fsx (1,1)-(1,20) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.

In [10]:
let examplePep2 = getIsotopicPattern "LTYYTPDYVVR" 2


Error: input.fsx (1,19)-(1,37) typecheck error The value or constructor 'getIsotopicPattern' is not defined. Maybe you want one of the following:
   getCvOfReplicates
   getStDevOfReplicates

In [11]:
plotIsotopicPattern FSharpAux.Colors.Table.Office.blue examplePep2.Pattern


Error: input.fsx (1,1)-(1,20) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.

## III. Simulation of isotopic patterns: revisited.

Now that we visualized the patterns of two sample peptides, we will simulate theoretical patterns
and compare them to the ones we measured! You will recognize a lot of the used code from *NB02c Isotopic distribution*
Note: we copy the code so you can make yourself familiar with it, of course we could also reference functions defined beforehand.


In [12]:
// create chemical formula for amino acid and add water to reflect hydrolysed state in mass spectrometer
let toFormula bioseq =  
    bioseq
    |> BioSeq.toFormula
    // peptides are hydrolysed in the mass spectrometer, so we add H2O
    |> Formula.add Formula.Table.H2O

let label n15LableEfficiency formula =
    let heavyN15 = Elements.Di  (Elements.createDi "N15" (Isotopes.Table.N15,n15LableEfficiency) (Isotopes.Table.N14,1.-n15LableEfficiency) )
    Formula.replaceElement formula Elements.Table.N heavyN15

// Predicts an isotopic distribution of the given formula at the given charge, 
// normalized by the sum of probabilities, using the MIDAs algorithm
let generateIsotopicDistribution (charge:int) (f:Formula.Formula) =
    IsotopicDistribution.MIDA.ofFormula 
        IsotopicDistribution.MIDA.normalizeByProbSum
        0.01
        0.000000001
        charge
        f

type SimulatedIsoPattern = 
    {|
        PeptideSequence : string
        Charge          : int
        LableEfficiency : float
        SimPattern      : list<(float*float)>
    |}

let simulateFrom peptideSequence charge lableEfficiency =
    let simPattern =
        peptideSequence
        |> BioSeq.ofAminoAcidString
        |> toFormula 
        |> label lableEfficiency
        |> generateIsotopicDistribution charge     
    {|
        PeptideSequence = peptideSequence
        Charge          = charge
        LableEfficiency = lableEfficiency
        SimPattern      = simPattern
    |}


In [13]:
let examplePep2_Sim1 = simulateFrom "LTYYTPDYVVR" 2 0.95

plotIsotopicPattern FSharpAux.Colors.Table.Office.orange examplePep2_Sim1.SimPattern


Error: input.fsx (3,1)-(3,20) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.

In [14]:
let examplePep2_Sim2 = simulateFrom "LTYYTPDYVVR" 2 0.99

plotIsotopicPattern FSharpAux.Colors.Table.Office.orange examplePep2_Sim2.SimPattern


Error: input.fsx (3,1)-(3,20) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.

## IV. Comparing measured and theoretical isotopic patterns.

As we see, there is a discrepancy between real and simulated patterns, both in peak height and in peak count. 
But before we compare both patterns, we have to take some things into consideration.
While both patterns are normalized in a way that their intensities
sum to 1., they were normalized independently from each other. Since it is often not possible to 
extract all peaks of an isotopic pattern from a MS run (e.g. due to measurement inaccuracies), we have to 
write a function which filters the simulated patterns for those peaks present in the experimentally 
measured one. Then we normalize it again and have two spectra that can be compared.
// How are distributions called that sum up to 1?


In [15]:
let normBySum (a:seq<float*float>) =
    let s = Seq.sumBy snd a 
    Seq.map (fun (x,y) -> x,y / s) a

let compareIsotopicDistributions (measured:ExtractedIsoPattern) (simulated:SimulatedIsoPattern)= 
    let patternSim' = 
        measured.Pattern 
        |> Seq.map (fun (mz,intensities) -> 
                mz,
                simulated.SimPattern
                |> Seq.filter (fun (mzSim,intensitiesSim) -> abs(mzSim-mz) < 0.05 )
                |> Seq.sumBy snd
            )
        |> normBySum
    {|
        Plot = 
            [
            plotIsotopicPattern FSharpAux.Colors.Table.Office.blue measured.Pattern
            plotIsotopicPattern FSharpAux.Colors.Table.Office.orange patternSim'
            ]
            |> Chart.combine
    |}


Error: input.fsx (5,44)-(5,63) typecheck error The type 'ExtractedIsoPattern' is not defined. Maybe you want one of the following:
   SimulatedIsoPattern
input.fsx (7,9)-(7,25) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (18,13)-(18,32) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.
input.fsx (19,13)-(19,32) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.

In [16]:
let comp1 = compareIsotopicDistributions examplePep2 examplePep2_Sim1
comp1.Plot


Error: input.fsx (1,13)-(1,41) typecheck error The value or constructor 'compareIsotopicDistributions' is not defined. Maybe you want one of the following:
   ComparisonConditionalOnAttribute
   CompilationRepresentationAttribute
   CompilationSourceNameAttribute
   CompilationArgumentCountsAttribute
   CompilerMessageAttribute
input.fsx (2,1)-(2,11) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.

In [17]:
let comp2 = compareIsotopicDistributions examplePep2 examplePep2_Sim2
comp2.Plot


Error: input.fsx (1,13)-(1,41) typecheck error The value or constructor 'compareIsotopicDistributions' is not defined. Maybe you want one of the following:
   ComparisonConditionalOnAttribute
   CompilationRepresentationAttribute
   CompilationSourceNameAttribute
   CompilationArgumentCountsAttribute
   CompilerMessageAttribute
input.fsx (2,1)-(2,11) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.

Comparing both simulations, we see that the simulation with a label efficiency of 0.99 fits the measured spectra better than the simulation with 0.95.
But since we do not want to find a better fit, but the best fit to our measured pattern, this is no goal that is achievable in a feasable way 
using visual inspections. As a solution we utilize the fact that isotopic patterns can be abstracted as ___ ___ (See: How are distributions called that sum up to 1?) distributions.
A measure to compare measured and theoretical distributions is the kullback leibler divergence. The following code block extends the function 
'compareIsotopicDistributions' to compute the KL divergence between the precisely measured distribution p and our approximation 
of p (q) using the mida algorithm. 


In [18]:
/// Calculates the Kullback-Leibler divergence Dkl(p||q) from q (theory, model, description, or approximation of p) 
/// to p (the "true" distribution of data, observations, or a ___ ___ precisely measured).
let klDiv (p:seq<float>) (q:seq<float>) = 
    Seq.fold2 (fun acc p q -> (System.Math.Log(p/q)*p) + acc ) 0. p q

let compareIsotopicDistributions' (measured:ExtractedIsoPattern) (simulated:SimulatedIsoPattern)= 
    let patternSim' = 
        measured.Pattern 
        |> Seq.map (fun (mz,intensities) -> 
                mz,
                simulated.SimPattern
                |> Seq.filter (fun (mzSim,intensitiesSim) -> abs(mzSim-mz) < 0.05 )
                |> Seq.sumBy snd
            )
        |> normBySum
    let klDiv = klDiv (patternSim' |> Seq.map snd)  (measured.Pattern |> Seq.map snd)
    {|
        KLDiv = klDiv
        Plot  = 
            [
            plotIsotopicPattern FSharpAux.Colors.Table.Office.blue measured.Pattern
            plotIsotopicPattern FSharpAux.Colors.Table.Office.orange patternSim'
            ]
            |> Chart.combine
    |}


Error: input.fsx (6,45)-(6,64) typecheck error The type 'ExtractedIsoPattern' is not defined. Maybe you want one of the following:
   SimulatedIsoPattern
input.fsx (8,9)-(8,25) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (15,12)-(15,21) typecheck error The value or constructor 'normBySum' is not defined.
input.fsx (16,54)-(16,70) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (21,13)-(21,32) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.
input.fsx (22,13)-(22,32) typecheck error The value or constructor 'plotIsotopicPattern' is not defined.

## V. Determining the lable efficiency: an optimiziation problem.

Using this function we can now visualize the kullback leibler divergence between
different models and the two peptides we measured. Since the lower the divergence. We will
also visualize the pattern with the best fit. Please inspect the Chart created by 'Chart.Point(lableEfficiency,comparison |> Seq.map (fun x -> x.KLDiv))' 
and write correct descriptions for the x and the y axis. (Fill: |> Chart.withX_AxisStyle "" and |> Chart.withY_AxisStyle "")


In [19]:
let lableEfficiency, comparison = 
    [|0.95 .. 0.001 .. 0.999|]
    |> Array.map (fun lableEfficiency -> 
            let sim = simulateFrom "DTDILAAFR" 2 lableEfficiency
            let comp = compareIsotopicDistributions' examplePep1 sim
            lableEfficiency,
            comp
        )
    |> Seq.unzip
let bestFit = comparison |> Seq.minBy (fun x -> x.KLDiv) 

Chart.Point(lableEfficiency,comparison |> Seq.map (fun x -> x.KLDiv))
|> Chart.withXAxisStyle ""
|> Chart.withYAxisStyle ""


Error: input.fsx (5,24)-(5,53) typecheck error The value or constructor 'compareIsotopicDistributions'' is not defined. Maybe you want one of the following:
   ComparisonConditionalOnAttribute
   CompilationRepresentationAttribute
   CompilationArgumentCountsAttribute
   CompilationSourceNameAttribute
   CompilerMessageAttribute
input.fsx (10,49)-(10,56) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (12,61)-(12,68) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.

In [20]:
bestFit.Plot


Error: input.fsx (1,1)-(1,8) typecheck error The value, namespace, type or module 'bestFit' is not defined.

In [21]:
let lableEfficiency2, comparison2 = 
    [|0.95 .. 0.001 .. 0.999|]
    |> Array.map (fun lableEfficiency -> 
            let sim = simulateFrom "LTYYTPDYVVR" 2 lableEfficiency
            let comp = compareIsotopicDistributions' examplePep2 sim
            lableEfficiency,
            comp
        )
    |> Seq.unzip

let bestFit2 = comparison2 |> Seq.minBy (fun x -> x.KLDiv) 

Chart.Point(lableEfficiency2,comparison2 |> Seq.map (fun x -> x.KLDiv))
|> Chart.withXAxisStyle ""
|> Chart.withYAxisStyle ""


Error: input.fsx (5,24)-(5,53) typecheck error The value or constructor 'compareIsotopicDistributions'' is not defined. Maybe you want one of the following:
   ComparisonConditionalOnAttribute
   CompilationRepresentationAttribute
   CompilationArgumentCountsAttribute
   CompilationSourceNameAttribute
   CompilerMessageAttribute
input.fsx (11,51)-(11,58) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (13,63)-(13,70) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.

In [22]:
bestFit2.Plot


Error: input.fsx (1,1)-(1,9) typecheck error The value, namespace, type or module 'bestFit2' is not defined.

Observing the output, we can make two observations: the function x(lablefficiency) = KL(measured,sim(lableeffciency)) has in both cases a local minimum
that is similar, yet slightly different for peptides "LTYYTPDYVVR" and "DTDILAAFR", and that the best fit resembles the measured distribution closely, but not
perfectly, what is the reason for this?

Finding this local minimum will give us the best estimator for the lable efficiency. This can be done using brute force approaches (as we just did) 
or more elaborate optimization techniques. For this we will use an algorithm called 'Brent's method'. This method is more precise and speeds up the calculation time (Why?). 
How close are the estimates?


In [23]:
let calcKL peptideSequence charge lableEfficiency = 
    let measured = 
        getIsotopicPattern peptideSequence charge
    let sim = simulateFrom peptideSequence charge lableEfficiency
    let comp = 
        compareIsotopicDistributions' measured sim
    comp.KLDiv

let est1 = Optimization.Brent.minimize (calcKL "DTDILAAFR" 2) 0.98 0.999
let est2 = Optimization.Brent.minimize (calcKL "LTYYTPDYVVR" 2) 0.98 0.999


Error: input.fsx (3,9)-(3,27) typecheck error The value or constructor 'getIsotopicPattern' is not defined. Maybe you want one of the following:
   getCvOfReplicates
   getStDevOfReplicates
input.fsx (6,9)-(6,38) typecheck error The value or constructor 'compareIsotopicDistributions'' is not defined. Maybe you want one of the following:
   ComparisonConditionalOnAttribute
   CompilationRepresentationAttribute
   CompilationArgumentCountsAttribute
   CompilationSourceNameAttribute
   CompilerMessageAttribute
input.fsx (7,5)-(7,15) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.

Since the estimates have a certain level of uncertainty we will repeat the estimation for 
some high intensity peptides and visualize the results. Please fill the x axis description (|> Chart.withX_AxisStyle "")


In [24]:
let highIntensityPeptides = 
    heavy
    |> Frame.getCol "20210312BN2_U1" 
    |> Series.sortBy (fun (x:float) -> - x)
    |> Series.filter (fun k v -> k.StringSequence |> String.exists (fun x -> x='[') |> not)

let estimates = 
    highIntensityPeptides
    |> Series.take 20 
    |> Series.map (fun k v -> 
        FSharp.Stats.Optimization.Brent.minimize (calcKL k.StringSequence k.Charge) 0.98 0.999
        )
    |> Series.values
    |> Seq.choose id


Error: input.fsx (2,5)-(2,10) typecheck error The value or constructor 'heavy' is not defined. Maybe you want one of the following:
   Navy
input.fsx (5,34)-(5,50) typecheck error Lookup on object of indeterminate type based on information prior to this program point. A type annotation may be needed prior to this program point to constrain the type of the object. This may allow the lookup to be resolved.
input.fsx (11,51)-(11,57) typecheck error The value or constructor 'calcKL' is not defined.

In [25]:
Chart.BoxPlot estimates
|> Chart.withXAxisStyle ""


Error: input.fsx (1,15)-(1,24) typecheck error The value or constructor 'estimates' is not defined. Maybe you want one of the following:
   Stats
   stats
   Stats
input.fsx (1,1)-(1,24) typecheck error A unique overload for method 'BoxPlot' could not be determined based on type information prior to this program point. A type annotation may be needed.

Known type of argument: 'a

Candidates:
 - static member Chart.BoxPlot: [<Runtime.InteropServices.Optional; Runtime.InteropServices.DefaultParameterValue ((null :> obj))>] ?x: #IConvertible seq * [<Runtime.InteropServices.Optional; Runtime.InteropServices.DefaultParameterValue ((null :> obj))>] ?y: #IConvertible seq * [<Runtime.InteropServices.Optional; Runtime.InteropServices.DefaultParameterValue ((null :> obj))>] ?Name: string * [<Runtime.InteropServices.Optional; Runtime.InteropServices.DefaultParameterValue ((null :> obj))>] ?ShowLegend: bool * [<Runtime.InteropServices.Optional; Runtime.InteropServices.DefaultParameterValue ((null 

Now that we know more than an educated guess of an lable efficiency estimate we can start with our main goal:
the absolute quantification of chlamydomonas proteins!
